# 🚀 YOLO Master Training — Enterprise-Grade (2,576 Images)
**Train Fresh from Clean Base Weights (Nano or Medium)**
- **Nano Model (`yolov8n.pt`)**: **~6 MB file size** — perfect for forwarding/sharing, instant AWS Lambda/edge deployment, trains in only **~8 to 10 minutes**!
- **Medium Model (`yolov8m.pt`)**: **~52 MB file size** — maximum capacity for 150+ dense micro-pipe cartons.
- **Unified Master Dataset**: 2,576 images (2,280 Train + 296 Val)
- **High-Resolution 1024px**: Ensures even the Nano model clearly resolves fine pipe contours and dark inner bores.


In [ ]:
# ── Cell 1: Environment & GPU Check ──────────────────────────────────────
!nvidia-smi
!pip install ultralytics --quiet

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ── Cell 2: Extract & Verify Clean Master Dataset ────────────────────────
import os, zipfile

DATASET_ZIP = '/content/pipe_dataset_colab.zip'
DATASET_DIR = '/content/pipe_dataset'

assert os.path.exists(DATASET_ZIP), f"❌ Missing {DATASET_ZIP}! Upload pipe_dataset_colab.zip to /content/"

print('Extracting master dataset...')
with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    zf.extractall('/content/')

train_imgs = len(os.listdir(f'{DATASET_DIR}/images/train'))
val_imgs   = len(os.listdir(f'{DATASET_DIR}/images/val'))

train_lbls = os.listdir(f'{DATASET_DIR}/labels/train')
neg_train  = sum(1 for f in train_lbls if os.path.getsize(f'{DATASET_DIR}/labels/train/{f}') == 0)
pos_train  = train_imgs - neg_train

val_lbls   = os.listdir(f'{DATASET_DIR}/labels/val')
neg_val    = sum(1 for f in val_lbls if os.path.getsize(f'{DATASET_DIR}/labels/val/{f}') == 0)
pos_val    = val_imgs - neg_val

print('\n✅ Master Dataset successfully verified:')
print(f'   Train images: {train_imgs} ({pos_train} Positive Pipes + {neg_train} Hard Negatives)')
print(f'   Val images  : {val_imgs} ({pos_val} Positive Pipes + {neg_val} Hard Negatives)')
print(f'   Total images: {train_imgs + val_imgs}')


In [ ]:
# ── Cell 3: Master Training (Nano vs Medium) ─────────────────────────────
from ultralytics import YOLO
import time

# ==============================================================================
# 🎯 SELECT YOUR MODEL ARCHITECTURE:
# ------------------------------------------------------------------------------
# 'yolov8m.pt'  -> Medium Model (~52 MB) : Maximum accuracy for dense 100+ pipe cartons (RECOMMENDED)
# 'yolov8n.pt'  -> Nano Model (~6 MB)    : Ultra-fast training (10 mins), lightweight deployment
# ==============================================================================
MODEL_CHOICE = 'yolov8m.pt'  # Change to 'yolov8n.pt' if you prefer ultra-small file size

# ==============================================================================
# ⏱️ TRAINING DURATION & EARLY STOPPING:
# ------------------------------------------------------------------------------
# TOTAL_EPOCHS : Total passes over all 2,280 training images (Default: 35)
# PATIENCE     : Early stopping threshold. If accuracy doesn't improve for 15
#                epochs, training stops to save time. (Set to 0 to train all 35 epochs)
# ==============================================================================
TOTAL_EPOCHS = 35
PATIENCE     = 15   # Set to 0 to disable early stopping completely

BATCH_SIZE   = 8 if 'm.pt' in MODEL_CHOICE else 16

print(f'🔥 Initializing Fresh Training with {MODEL_CHOICE}...')
model = YOLO(MODEL_CHOICE)

print(f'   Architecture : {MODEL_CHOICE}')
print(f'   Dataset      : 2,280 Train (2,576 Total Master Images)')
print(f'   Validation   : 296 Val')
print(f'   Resolution   : 1024x1024')
print(f'   Batch Size   : {BATCH_SIZE}')
print(f'   Total Epochs : {TOTAL_EPOCHS}')
print(f'   Patience     : {PATIENCE} (Early stopping after {PATIENCE} stagnant epochs)')

start_time = time.time()

results = model.train(
    data         = f'{DATASET_DIR}/data.yaml',
    epochs       = TOTAL_EPOCHS,
    patience     = PATIENCE,
    imgsz        = 1024,
    batch        = BATCH_SIZE,
    device       = 0,
    workers      = 2,
    max_det      = 1500,

    # Transfer learning schedule
    lr0          = 0.01,
    lrf          = 0.01,
    warmup_epochs= 3,
    cos_lr       = True,

    # Augmentations for robust real-world generalization
    mosaic       = 0.5,
    close_mosaic = 5,
    mixup        = 0.05,
    copy_paste   = 0.0,
    fliplr       = 0.5,
    flipud       = 0.0,
    degrees      = 8.0,
    shear        = 3.0,
    scale        = 0.35,
    hsv_h        = 0.015,
    hsv_s        = 0.5,
    hsv_v        = 0.4,

    # Loss weights balanced for pipe detection and background rejection
    box          = 7.5,
    cls          = 1.5,
    dfl          = 1.5,

    project      = '/content/runs',
    name         = 'pipe_enterprise',
    exist_ok     = True,
    plots        = True,
    save         = True,
    save_period  = 5,
    verbose      = True
)

total_min = (time.time() - start_time) / 60.0
print(f'\n⏱️ Training finished in {total_min:.1f} minutes!')


In [ ]:
# ── Cell 4: Accuracy Benchmark (Positive Pipes & Negative Zero-Check) ─────
import os, math
import numpy as np
from pathlib import Path
from ultralytics import YOLO

BEST_PATH = '/content/runs/pipe_enterprise/weights/best.pt'
DATA_YAML = '/content/pipe_dataset/data.yaml'

assert os.path.exists(BEST_PATH), f"❌ Missing {BEST_PATH}"

print("🔍 Evaluating trained model on validation set...")
model = YOLO(BEST_PATH)
val_res = model.val(data=DATA_YAML, imgsz=1024, batch=8, conf=0.25, iou=0.45, max_det=1500, verbose=False)

print("\n" + "═"*70)
print("              📊 MODEL PERFORMANCE REPORT")
print("═"*70)
print(f"  mAP@50       : {float(val_res.box.map50)*100:>6.2f}%")
print(f"  mAP@50-95    : {float(val_res.box.map)*100:>6.2f}%")
print(f"  Precision    : {float(val_res.box.mp)*100:>6.2f}%")
print(f"  Recall       : {float(val_res.box.mr)*100:>6.2f}%")
print("═"*70)

def filter_concentric(boxes):
    if not boxes: return []
    sorted_b = sorted(boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]), reverse=True)
    supp = set()
    n = len(sorted_b)
    for i in range(n):
        if i in supp: continue
        b1 = sorted_b[i]
        w1, h1 = max(1.0, b1[2]-b1[0]), max(1.0, b1[3]-b1[1])
        c1x, c1y = (b1[0]+b1[2])/2.0, (b1[1]+b1[3])/2.0
        max_d1 = max(w1, h1)
        for j in range(i+1, n):
            if j in supp: continue
            b2 = sorted_b[j]
            w2, h2 = max(1.0, b2[2]-b2[0]), max(1.0, b2[3]-b2[1])
            ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
            ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
            inter = max(0.0, ix2-ix1) * max(0.0, iy2-iy1)
            ios = inter / (w2*h2)
            dist = math.hypot(c1x - (b2[0]+b2[2])/2.0, c1y - (b2[1]+b2[3])/2.0)
            if (ios >= 0.45 and dist < max_d1*0.48) or ios >= 0.70:
                supp.add(j)
    return [b for idx, b in enumerate(sorted_b) if idx not in supp]

print("\n📦 1. REAL INSUFLEX CARTON ACCURACY (Positive Tests):")
for cname, gt in [('insuflex_carton_thick.jpg', 67), ('insuflex_carton_thin.jpg', 158)]:
    cp = f'/content/pipe_dataset/images/train/{cname}'
    if os.path.exists(cp):
        r = model.predict(source=cp, conf=0.25, imgsz=1024, max_det=1500, verbose=False)[0]
        cnt = len(filter_concentric([b.xyxy[0].tolist() for b in r.boxes]))
        acc = (1.0 - abs(cnt - gt)/float(gt)) * 100.0
        print(f"  {cname:<28} | Ground Truth: {gt:>3} | Count: {cnt:>3} | Accuracy: {acc:6.2f}%")

print("\n🚫 2. NEGATIVE IMAGE ZERO-DETECTION VERIFICATION (Target: 0 Pipes Detected):")
neg_images = [
    ('neg_user_construction.jpg', 'Construction site & workers helmets'),
    ('neg_user_car.jpg', 'Car driver with seatbelt'),
    ('neg_cardboard_punch_holes_val.jpg', 'Cardboard punch holes & flap slots'),
    ('neg_user_hand_toothbrush_val.jpg', 'Hand holding toothbrush head'),
    ('neg_user_puja_bananas_fruits_val.jpg', 'Puja bananas & fruit baskets'),
    ('neg_user_desk_bottles_room_clutter_val.jpg', 'Room desk bottles & thermos cap'),
    ('neg_user_crate_bushings_screws_val.jpg', 'Crate of screws & bushings'),
    ('neg_worker_head_face_val.jpg', 'Worker head and face'),
    ('neg_user_desk.jpg', 'Office desk & laptop mouse'),
]

for fname, desc in neg_images:
    found = False
    for split in ['val', 'train']:
        np_path = f'/content/pipe_dataset/images/{split}/{fname}'
        if os.path.exists(np_path):
            r_new = len(model.predict(source=np_path, conf=0.25, imgsz=1024, verbose=False)[0].boxes)
            status = '✅ PASS (0 detected)' if r_new == 0 else f'⚠️ {r_new} detected'
            print(f"  {desc:<36} | Detected: {r_new:>2} FPs | {status}")
            found = True
            break


In [ ]:
# ── Cell 5: Export to ONNX ────────────────────────────────────────────────
from ultralytics import YOLO
import os

BEST_WEIGHTS = '/content/runs/pipe_enterprise/weights/best.pt'
best_model = YOLO(BEST_WEIGHTS)

print('Exporting best model to ONNX format (opset=11, imgsz=1024)...')
best_model.export(
    format   = 'onnx',
    imgsz    = 1024,
    opset    = 11,
    simplify = True,
    dynamic  = False
)

BEST_ONNX = BEST_WEIGHTS.replace('.pt', '.onnx')
print(f'\n✅ ONNX Export Successful: {BEST_ONNX}')
print(f'   Size: {os.path.getsize(BEST_ONNX)/1e6:.1f} MB')


In [ ]:
# ── Cell 6: Download Model Files to Your PC ───────────────────────────────
import shutil, os
from google.colab import files

BEST_WEIGHTS = '/content/runs/pipe_enterprise/weights/best.pt'
BEST_ONNX    = BEST_WEIGHTS.replace('.pt', '.onnx')

suffix = 'n' if 'yolov8n' in str(BEST_WEIGHTS) or os.path.getsize(BEST_WEIGHTS) < 20e6 else 'm'
pt_name   = f'best_insulation_pipe_{suffix}_1024.pt'
onnx_name = f'best_insulation_pipe_{suffix}_1024.onnx'

target_pt   = f'/content/{pt_name}'
target_onnx = f'/content/{onnx_name}'

shutil.copy(BEST_WEIGHTS, target_pt)
shutil.copy(BEST_ONNX,    target_onnx)

print(f'⬇️ Downloading {pt_name} ({os.path.getsize(target_pt)/1e6:.1f} MB)...')
files.download(target_pt)

print(f'⬇️ Downloading {onnx_name} ({os.path.getsize(target_onnx)/1e6:.1f} MB)...')
files.download(target_onnx)

print('\n🎉 Done! When downloaded, place the files into:')
print('   C:\\AI\\backend\\models\\')
